In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
df = pd.read_parquet('../data/processed/feature_store.parquet')
print(f"Loaded feature store: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(2)

In [ ]:

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='neighbourhood_group_cleansed', y='price', showfliers=False, palette='Set2')
plt.title('Listing Price Distribution by NYC Borough')
plt.xlabel('Borough')
plt.ylabel('Price ($/night)')
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(9, 8))
scatter = plt.scatter(
    df['longitude'], df['latitude'], 
    c=df['log_price'], cmap='viridis', 
    alpha=0.4, s=6
)
plt.colorbar(scatter, label='Log(Price)')
plt.title('Geospatial Price Density Map of NYC Listings')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:

df['subway_bin'] = pd.cut(
    df['distance_to_nearest_subway_km'], 
    bins=[0, 0.25, 0.5, 0.75, 1.0, 2.0, 5.0], 
    labels=['<250m', '250-500m', '500-750m', '750m-1km', '1-2km', '>2km']
)

subway_price = df.groupby('subway_bin', observed=False)['price'].median().reset_index()

plt.figure(figsize=(9, 4))
sns.barplot(data=subway_price, x='subway_bin', y='price', color='royalblue')
plt.title('Median Listing Price by Distance to Nearest Subway Station')
plt.xlabel('Distance to Nearest Subway')
plt.ylabel('Median Price ($)')
plt.tight_layout()
plt.show()

In [ ]:

numeric_cols = [
    'price', 'log_price', 'accommodates', 'bedrooms', 'beds', 
    'bathrooms', 'amenity_count', 'distance_to_nearest_subway_km', 
    'subway_count_1km', 'number_of_reviews'
]

plt.figure(figsize=(9, 7))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Correlation Matrix of Key Features')
plt.tight_layout()
plt.show()

In [ ]:

features = [
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'is_shared_bath',
    'amenity_count', 'distance_to_nearest_subway_km', 'subway_count_1km',
    'number_of_reviews', 'review_scores_rating', 'host_is_superhost'
]
cat_cols = ['room_type', 'neighbourhood_group_cleansed']

X = pd.get_dummies(df[features + cat_cols], columns=cat_cols, drop_first=True)
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"Baseline Random Forest Performance:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(9, 4))
importances.plot(kind='barh', color='teal').invert_yaxis()
plt.title('Top 10 Feature Importances in Price Prediction')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()